In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
absl.logging.use_absl_handler()

import random
import warnings
import json
import threading
import gc
import pickle
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import pytz
import xgboost as xgb
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks, regularizers

tf.keras.backend.clear_session()
gc.collect()

# ── GPU Mixed Precision ─────────────────────────────────────────────────────
policy = keras.mixed_precision.Policy('float32')   # ← float32: stable on Keras 3.x
keras.mixed_precision.set_global_policy(policy)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── GPU Setup ──────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    try:
        d = tf.config.experimental.get_device_details(gpus[0])
        print("GPU:", d.get('device_name', gpus[0].name))
    except Exception:
        print("GPU:", gpus[0].name)
    print(f"Precision policy: {policy.name}")
else:
    print("WARNING: No GPU detected — training will be slow.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
IST = pytz.timezone('Asia/Kolkata')
_LOCK = threading.Lock()

def tprint(*a, **kw):
    with _LOCK: print(*a, **kw)

OUTPUT_DIR = '/kaggle/working/output_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

XGB_VERSION = tuple(map(int, xgb.__version__.split('.')[:2]))
tprint(f"XGBoost {xgb.__version__} | TF {tf.__version__} | Policy: {policy.name}")

/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/.gitignore
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/to-do.md
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/README.md
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/CHT.ipynb
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/DOCUMENTATION.md
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/results.md
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset/Table_datasheet/manufactuer_specifications.xlsx
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset/NCA/25degC/data_analysis_NCA_25degC.m
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset/NCA/25degC/NCA_k3_2C_25degC.xlsx
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset/NCA/25degC/NCA_k6_3C_25degC.xlsx
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset/NCA/25degC/NCA_k3_0_05C_25degC.xlsx
/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/v

E0000 00:00:1781553842.999482      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781553843.110648      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781553844.086828      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781553844.086885      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781553844.086888      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781553844.086890      23 computation_placer.cc:177] computation placer already registered. Please check linka

GPU: Tesla T4
Precision policy: float32
XGBoost 3.2.0 | TF 2.19.0 | Policy: float32


# PCM Thermal Hybrid — Model 1 v2.0 (Kaggle-Fast \| Clean-Target \| Interrupt-Safe)

-   PRETRAIN Single dataset: RT-45 AR=0.4 only (\~3× faster than all-PCM
    concat). Epochs 80, patience 15. \~5–8 min total.

-   SEQ_LEN 25 steps (was 75). Cuts sequence construction & GRU cost by
    3×. Still covers the relevant thermal time constant.

-   VERBOSE Progress bars enabled (verbose=1) so Kaggle cell shows live
    epochs.

-   CACHE True on tf.data — first epoch caches to RAM, subsequent epochs
    fly. Safe because SEQ_LEN=25 keeps tensor memory modest.

-   DROPOUT Kept high (0.20) as requested; L2=5e-3; float32;
    clean-target eval.

# ─────────────────────────────────────────────────────────────────────────────

# PATHS & REGISTRY

# ─────────────────────────────────────────────────────────────────────────────

In [2]:
DATA_BASE = '/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/data'
VAL_BASE  = '/kaggle/input/datasets/piyushkrgupta/model-dataset-v2/validation-dataset'
PCM_REGISTRY: Dict[str, Dict[str, str]] = {
    'RT-45': {
        '0.3': os.path.join(DATA_BASE, 'RT-45', '0.3.xlsx'),
        '0.4': os.path.join(DATA_BASE, 'RT-45', '0.4.xlsx'),
        '0.5': os.path.join(DATA_BASE, 'RT-45', '0.5.xlsx'),
    },
    'Lauric_Acid': {
        '0.3': os.path.join(DATA_BASE, 'laureic_acid', '0.3.xlsx'),
        '0.4': os.path.join(DATA_BASE, 'laureic_acid', '0.4.xlsx'),
        '0.5': os.path.join(DATA_BASE, 'laureic_acid', '0.5.xlsx'),
    },
    'Palmitic_Acid': {
        '0.3': os.path.join(DATA_BASE, 'palmuric_acid', '0.3.xlsx'),
        '0.4': os.path.join(DATA_BASE, 'palmuric_acid', '0.4.xlsx'),
        '0.5': os.path.join(DATA_BASE, 'palmuric_acid', '0.5.xlsx'),
    },
}

NCA_DATA_PATHS = {
    'k1': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k1_5C_25degC.xlsx'),
    'k2': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k2_5C_25degC.xlsx'),
    'k3': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k3_5C_25degC.xlsx'),
    'k4': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k4_5C_25degC.xlsx'),
    'k5': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k5_5C_25degC.xlsx'),
    'k6': os.path.join(VAL_BASE, 'NCA', '25degC', 'NCA_k6_5C_25degC.xlsx'),
}
MANUFACTURER_SPEC_PATH = os.path.join(VAL_BASE, 'Table_datasheet', 'manufactuer_specifications.xlsx')

# ─────────────────────────────────────────────────────────────────────────────

# PCM THERMOPHYSICAL PROPERTIES & CONFIGURATION

# ─────────────────────────────────────────────────────────────────────────────

In [3]:
PCM_PROPS: Dict[str, Dict] = {
    'RT-45': {
        'L_latent':   139_700.0,
        'Cp_liquid':    2_333.0,
        'Cp_solid':     3_028.0,
        'rho_liquid':     770.0,
        'rho_solid':      880.0,
        'mu_liquid':     0.0256,
        'k_liquid':      0.2415,
        'k_solid':       0.2415,
        'beta_liquid': 1.25e-4,
        'T_liquidus':    322.0,
        'T_solidus':     308.0,
    },
    'Lauric_Acid': {
        'L_latent':   187_210.0,
        'Cp_liquid':    2_390.0,
        'Cp_solid':     2_180.0,
        'rho_liquid':     885.0,
        'rho_solid':      940.0,
        'mu_liquid':     0.0080,
        'k_liquid':      0.1400,
        'k_solid':       0.1600,
        'beta_liquid': 8.00e-4,
        'T_liquidus':    321.35,
        'T_solidus':     316.65,
    },
    'Palmitic_Acid': {
        'L_latent':   203_400.0,
        'Cp_liquid':    2_480.0,
        'Cp_solid':     2_200.0,
        'rho_liquid':     853.0,
        'rho_solid':      853.0,
        'mu_liquid':     0.0078,
        'k_liquid':      0.2100,
        'k_solid':       0.2100,
        'beta_liquid': 7.72e-4,
        'T_liquidus':    334.14,
        'T_solidus':     332.84,
    },
}

for _pcm, _p in PCM_PROPS.items():
    _p['nu_liquid']    = _p['mu_liquid'] / _p['rho_liquid']
    _p['alpha_liquid'] = _p['k_liquid'] / (_p['rho_liquid'] * _p['Cp_liquid'])
    _p['Pr_liquid']    = _p['Cp_liquid'] * _p['mu_liquid'] / _p['k_liquid']

_DEFAULT_PROPS = PCM_PROPS['RT-45']

# ── External air cooling (Re = 1000) ────────────────────────────────────────
RE_AIR     = 1000.0
PR_AIR     = 0.713
K_AIR      = 0.02625
D_CELL     = 0.018
L_CELL     = 0.065
A_CELL     = np.pi * D_CELL * L_CELL
NU_EXT     = 0.228 * (RE_AIR ** 0.731) * (PR_AIR ** 0.36)
H_EXT      = NU_EXT * K_AIR / D_CELL
R_EXT      = 1.0 / (H_EXT * A_CELL)
L_PCM_HALF = 0.005

tprint(f"  External air: Nu_ext={NU_EXT:.2f}  h_ext={H_EXT:.1f} W/(m²·K)  R_ext={R_EXT:.3f} K/W")

PCM_LATENT_HEAT_MEAN = 176_770.0
PCM_MASS_KG          = 0.05
SENSOR_NOISE_STD = 0.8       # K — battery (regularization)
SENSOR_NOISE_PCM = 0.5       # K — PCM   (regularization)
PIMA_WINDOW      = 15

RAW_COLS   = ['time', 'aspect_ratio', 'liquid_frac', 'Nu', 'T_battery', 'T_pcm']
TARGET_COL = 'T_battery'
XGB_TARGET = 'dT_battery'

FEATURE_COLS_CORE = [
    'liquid_frac', 'Nu', 'T_pcm',
    'T_bat_lag3', 'dT_lag3', 'heat_flux_lag3',
    'melting_rate', 'T_pcm_rate',
    'stefan_number', 'nusselt_scaled', 'enthalpy_proxy',
    'phase_transition_zone',
    'Nu_ratio', 'Biot_ext', 'h_ext_norm',
]

FEATURE_COLS_EXT = FEATURE_COLS_CORE + [
    'lf_sq', 'Nu_x_lf', 'Nu_abs', 'thermal_res',
    'phase_flag', 'T_bat_rate_lag', 'Nu_rate', 'T_pcm_lf',
    'ste_proxy', 'aspect_ratio_num', 'delta_T',
    'Ra_scaled', 'convective_intensity',
]

LAG_STEPS     = [1, 3, 5, 10, 25]
LAG_BASE_COLS = [
    'T_bat_lag3', 'T_pcm', 'liquid_frac', 'Nu',
    'dT_lag3', 'heat_flux_lag3', 'melting_rate',
]

XGB_PARAMS = dict(
    n_estimators     = 900,
    learning_rate    = 0.025,
    max_depth        = 4,
    min_child_weight = 5,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    colsample_bylevel= 0.85,
    gamma            = 0.05,
    reg_alpha        = 0.3,
    reg_lambda       = 1.5,
    objective        = 'reg:squarederror',
    tree_method      = 'hist',
    device           = 'cuda',
    random_state     = SEED,
)

# v2.3: shorter sequence for speed
SEQ_STEPS    = 25
BATCH_SIZE   = 1024
EPOCHS       = 500
PATIENCE     = 60
TRAIN_RATIO  = 0.70
VAL_FRACTION = 0.15

MIN_R2_FOR_ENSEMBLE     = -1.0
FALLBACK_TO_BEST_SINGLE = True

T_AMBIENT_K        = 298.15
NCA_DISCHARGE_STEP = 5
OVERFIT_THRESHOLD = 0.999

def select_best_r2(met, thresh=OVERFIT_THRESHOLD):
    """Return (best_r2_value, source_label, is_overfit)."""
    r2c = float(met.get('r2', float('nan')))
    r2n = float(met.get('r2_noisy', float('nan')))
    c_over = not np.isnan(r2c) and r2c >= thresh
    n_over = not np.isnan(r2n) and r2n >= thresh
    if c_over and n_over:
        return None, 'overfit', True
    if c_over:
        return r2n, 'noisy', False
    if n_over:
        return r2c, 'clean', False
    if np.isnan(r2c) and np.isnan(r2n):
        return None, 'none', False
    if np.isnan(r2c):
        return r2n, 'noisy', False
    if np.isnan(r2n):
        return r2c, 'clean', False
    if r2c >= r2n:
        return r2c, 'clean', False
    else:
        return r2n, 'noisy', False

def is_appropriate(met, thresh=OVERFIT_THRESHOLD):
    best_r2, _, is_over = select_best_r2(met, thresh)
    if is_over or best_r2 is None:
        return False
    return best_r2 > 0.0

  External air: Nu_ext=31.48  h_ext=45.9 W/(m²·K)  R_ext=5.926 K/W


# ─────────────────────────────────────────────────────────────────────────────

# DATA PIPELINE

# ─────────────────────────────────────────────────────────────────────────────

In [4]:
def load_raw(path: str) -> pd.DataFrame:
    df = (pd.read_excel(path, engine='openpyxl') if path.endswith('.xlsx')
          else pd.read_csv(path, encoding='latin1'))
    missing = set(RAW_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    return df.sort_values('time').reset_index(drop=True)

def clean(df: pd.DataFrame, tag: str = '') -> pd.DataFrame:
    d = df.copy()
    dupes = d.duplicated(subset=['time'])
    if dupes.sum():
        tprint(f"  [{tag}] Dropping {dupes.sum()} duplicate time-steps")
        d = d[~dupes]
    if d.isnull().sum().sum():
        d = d.ffill().bfill()
    d['liquid_frac'] = d['liquid_frac'].clip(0.0, 1.0)
    q1, q3 = d['Nu'].quantile(0.05), d['Nu'].quantile(0.95)
    iqr = q3 - q1
    d['Nu'] = d['Nu'].clip(q1 - 5*iqr, q3 + 5*iqr)
    return d.sort_values('time').reset_index(drop=True)

def engineer_features(df: pd.DataFrame, ar_label: str, pcm_name: str) -> pd.DataFrame:
    d     = df.copy()
    props = PCM_PROPS.get(pcm_name, _DEFAULT_PROPS)
    L     = props['L_latent']
    Cp    = props['Cp_liquid']
    T_liq = props['T_liquidus']
    T_sol = props['T_solidus']
    beta  = props['beta_liquid']
    Pr    = props['Pr_liquid']
    k     = props['k_liquid']
    nu    = props['nu_liquid']
    alpha = props['alpha_liquid']

    d['delta_T']        = d['T_battery'] - d['T_battery'].iloc[0]
    d['T_bat_lag3']     = d['T_battery'].shift(3).fillna(d['T_battery'].iloc[0])
    T_pcm_lag3          = d['T_pcm'].shift(3).fillna(d['T_pcm'].iloc[0])
    d['dT_lag3']        = d['T_bat_lag3'] - T_pcm_lag3
    Nu_lag3             = d['Nu'].shift(3).fillna(d['Nu'].iloc[0])
    d['heat_flux_lag3'] = Nu_lag3 * d['dT_lag3']
    d['melting_rate']   = d['liquid_frac'].diff().fillna(0.0)
    d['T_pcm_rate']     = d['T_pcm'].diff().fillna(0.0)
    d[XGB_TARGET]       = d[TARGET_COL].diff(3).fillna(0.0)

    d['lf_sq']            = d['liquid_frac'] ** 2
    d['Nu_x_lf']          = d['Nu'] * d['liquid_frac']
    d['Nu_abs']           = d['Nu'].abs()
    d['thermal_res']      = d['dT_lag3'] / (d['Nu_abs'] + 1e-6)
    d['T_bat_rate_lag']   = d['T_bat_lag3'].diff().fillna(0.0)
    d['Nu_rate']          = d['Nu'].diff().fillna(0.0)
    d['T_pcm_lf']         = d['T_pcm'] * d['liquid_frac']
    d['aspect_ratio_num'] = float(ar_label)
    d['ste_proxy']        = d['delta_T'] / (L / 1000.0 + 1e-6)
    d['phase_flag']       = 0
    d.loc[d['liquid_frac'] > 0.0,  'phase_flag'] = 1
    d.loc[d['liquid_frac'] >= 1.0, 'phase_flag'] = 2

    d['stefan_number']       = Cp * (d['T_pcm'] - T_liq).abs() / (L + 1e-6)
    d['nusselt_scaled']      = d['Nu'] / (Pr ** (1.0/3.0) + 1e-6)
    d['enthalpy_proxy']      = (
        d['liquid_frac'] * L / 1e6 +
        Cp * (d['T_pcm'] - T_sol).clip(lower=0.0) / 1e6
    )
    d['phase_transition_zone'] = 1.0 - np.abs(2.0 * d['liquid_frac'] - 1.0)

    L_char = 0.01 * float(ar_label)
    delta_T_conv = (d['T_battery'] - d['T_pcm']).abs()
    Ra = (9.81 * beta * delta_T_conv * L_char**3) / (nu * alpha + 1e-30)
    d['Ra_scaled']            = np.log1p(Ra.clip(lower=0.0))
    d['convective_intensity'] = d['Ra_scaled'] * d['liquid_frac']

    d['Nu_ratio']   = d['Nu'] / (NU_EXT + 1e-6)
    d['Biot_ext']   = H_EXT * L_PCM_HALF / (k + 1e-6)
    d['h_ext_norm'] = H_EXT / 100.0

    d['T_battery_clean'] = d['T_battery'].copy()
    d['T_pcm_clean']     = d['T_pcm'].copy()

    np.random.seed(SEED)
    d['T_battery'] = d['T_battery'] + np.random.normal(0, SENSOR_NOISE_STD, len(d))
    d['T_pcm']     = d['T_pcm']     + np.random.normal(0, SENSOR_NOISE_PCM, len(d))

    return d

def add_cumulative_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    if 'heat_flux_lag3' in d.columns:
        d['cumul_heat'] = d['heat_flux_lag3'].cumsum()
    if 'melting_rate' in d.columns:
        d['cumul_melt'] = d['melting_rate'].cumsum().clip(0, 1)
    return d

def add_lag_rolling(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for col in LAG_BASE_COLS:
        if col not in d.columns:
            continue
        for lag in LAG_STEPS:
            d[f'{col}_lag{lag}']       = d[col].shift(lag)
            d[f'{col}_rollmean_{lag}'] = d[col].shift(1).rolling(lag, min_periods=1).mean()
            d[f'{col}_rollstd_{lag}']  = d[col].shift(1).rolling(lag, min_periods=1).std().fillna(0)
            d[f'{col}_rollmax_{lag}']  = d[col].shift(1).rolling(lag, min_periods=1).max()
    return d

def get_xgb_cols() -> List[str]:
    cols = list(FEATURE_COLS_EXT)
    for col in LAG_BASE_COLS:
        for lag in LAG_STEPS:
            cols += [
                f'{col}_lag{lag}',
                f'{col}_rollmean_{lag}',
                f'{col}_rollstd_{lag}',
                f'{col}_rollmax_{lag}',
            ]
    cols += ['cumul_heat', 'cumul_melt']
    return cols
    
def temporal_split(df, ratio=TRAIN_RATIO):
    """
    v2.4 FINAL: Melt-centric split.
    Test set is anchored to the phase-change window (ramp-in + plateau + ramp-out).
    This guarantees:
      - Test set always has thermal variance (no NaN R²)
      - Skill vs Persist is evaluated on the most challenging dynamic regime
      - All ARs and PCMs are compared on equivalent physics
      - Training set contains both pre-melt solid and post-melt liquid for full coverage
    """
    d = df.copy().reset_index(drop=True)
    n = len(d)

    # ── Locate the melting window ──────────────────────────────────────────
    melt_mask = (d['liquid_frac'] > 0.1) & (d['liquid_frac'] < 0.9)
    melt_idx  = d.index[melt_mask].tolist()

    if len(melt_idx) > 10:
        m_start = melt_idx[0]
        m_end   = melt_idx[-1]
        m_len   = m_end - m_start

        # Pad 50 % of melt length on each side to capture ramp-in and early liquid
        pad        = max(int(m_len * 0.50), SEQ_STEPS + 5)
        test_start = max(0, m_start - pad)
        test_end   = min(n, m_end   + pad)

        # Clamp: test window must be 15–35 % of total dataset
        lo = int(n * 0.15)
        hi = int(n * 0.35)
        w  = test_end - test_start

        if w > hi:
            # Window too large: shrink symmetrically around melt centre
            centre     = (m_start + m_end) // 2
            half       = hi // 2
            test_start = max(0, centre - half)
            test_end   = min(n, centre + half)
        elif w < lo:
            # Window too small (NePCM / very fast melt): expand symmetrically
            centre     = (m_start + m_end) // 2
            half       = lo // 2
            test_start = max(0, centre - half)
            test_end   = min(n, centre + half)

    else:
        # No discernible plateau → middle 25 % of timeline
        test_start = int(n * 0.375)
        test_end   = int(n * 0.625)

    # ── Diagnostic print so you can verify the split per case ─────────────
    tprint(f"  [split] n={n}  test=[{test_start}:{test_end}]  "
           f"len={test_end-test_start}  "
           f"melt_steps={len(melt_idx)}")

    train_df = pd.concat(
        [d.iloc[:test_start], d.iloc[test_end:]]
    ).copy().reset_index(drop=True)
    test_df  = d.iloc[test_start:test_end].copy().reset_index(drop=True)
    return train_df, test_df

def build_sequences(df: pd.DataFrame, feat_cols: List[str], steps: int,
                    target_col: str = 'dT_target_scaled'):
    X   = df[feat_cols].values.astype(np.float32)
    y_t = df[target_col].values.astype(np.float32)
    y_l = df['lf_target_scaled'].values.astype(np.float32)
    if len(df) <= steps:
        raise ValueError(f"Too few rows ({len(df)}) for seq_steps={steps}")
    # Vectorized sliding window via advanced indexing — fast & correct shape (batch, steps, features)
    idx = np.arange(steps)[None, :] + np.arange(len(df) - steps)[:, None]
    Xs = X[idx]
    yt = y_t[steps:]
    yl = y_l[steps:]
    return Xs, yt, yl

def add_global_flags(df, pcm_name, ar):
    d = df.copy()
    d['is_rt45']     = 1.0 if 'RT-45'  in pcm_name else 0.0
    d['is_lauric']   = 1.0 if 'Lauric' in pcm_name else 0.0
    d['is_palmitic'] = 1.0 if 'Palmit' in pcm_name else 0.0
    d['ar_0.3'] = 1.0 if ar == '0.3' else 0.0
    d['ar_0.4'] = 1.0 if ar == '0.4' else 0.0
    d['ar_0.5'] = 1.0 if ar == '0.5' else 0.0
    return d

@dataclass
class DataBundle:
    xgb: Dict
    lstm: Dict
    test_df: pd.DataFrame
    train_df: pd.DataFrame
    pcm: str
    ar: str
    T_initial: float

# ─────GLOBAL SCALERS & LIGHTWEIGHT PRE-TRAINING
def fit_global_scalers(registry):
    tprint("  Fitting global scalers for TCN-GRU curriculum learning...")
    all_train_dfs = []
    for pcm_name, ar_map in registry.items():
        for ar, path in ar_map.items():
            if not os.path.exists(path):
                continue
            df = clean(load_raw(path), f"{pcm_name}_AR{ar}")
            df = engineer_features(df, ar, pcm_name)
            df = add_cumulative_features(df)
            df = add_lag_rolling(df)
            df = add_global_flags(df, pcm_name, ar)
            tr = df.iloc[:int(len(df) * TRAIN_RATIO)]
            all_train_dfs.append(tr)
    if not all_train_dfs:
        raise ValueError("No training data found for global scaler fitting.")
    combined = pd.concat(all_train_dfs, ignore_index=True)
    lstm_feats = list(FEATURE_COLS_CORE) + [
        'is_rt45', 'is_lauric', 'is_palmitic', 'ar_0.3', 'ar_0.4', 'ar_0.5'
    ]
    xs = {c: RobustScaler().fit(combined[[c]]) for c in lstm_feats}
    ys_t_delta = RobustScaler().fit(combined[[XGB_TARGET]])
    ys_l = MinMaxScaler().fit(combined[['liquid_frac']])
    tprint(f"  Global scalers fitted on {len(combined)} samples, {len(lstm_feats)} features")
    return xs, ys_t_delta, ys_l, lstm_feats

def pretrain_tcn_gru(registry, global_scalers, epochs=80, patience=15):
    xs, ys_t_delta, ys_l, lstm_feats = global_scalers
    tprint("\n" + "="*70 + "\n  TCN-GRU PRE-TRAINING on RT-45 AR=0.4 (single dataset)\n" + "="*70)

    path = registry.get('RT-45', {}).get('0.4')
    if not path or not os.path.exists(path):
        tprint("  [WARN] RT-45 AR=0.4 not found; skipping pre-training")
        return None

    df = clean(load_raw(path), "RT-45_AR0.4_pre")
    df = engineer_features(df, '0.4', 'RT-45')
    df = add_cumulative_features(df)
    df = add_lag_rolling(df)
    df = add_global_flags(df, 'RT-45', '0.4')

    n = len(df)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VAL_FRACTION)
    train_df = df.iloc[:n_tr - n_va].copy().reset_index(drop=True)
    val_df   = df.iloc[n_tr - n_va:n_tr].copy().reset_index(drop=True)

    def scale(d):
        d2 = d.copy()
        for c in lstm_feats:
            d2[c] = xs[c].transform(d2[[c]])
        d2['dT_target_scaled'] = ys_t_delta.transform(d2[[XGB_TARGET]])
        d2['lf_target_scaled'] = ys_l.transform(d2[['liquid_frac']])
        return d2

    Xtr, ytr_t, ytr_l = build_sequences(scale(train_df), lstm_feats, SEQ_STEPS,
                                        target_col='dT_target_scaled')
    Xva, yva_t, yva_l = build_sequences(scale(val_df), lstm_feats, SEQ_STEPS,
                                        target_col='dT_target_scaled')

    model = build_model(len(lstm_feats), SEQ_STEPS)
    tprint(f"  Pre-train model params: {model.count_params():,}")

    train_ds = make_dataset(Xtr, ytr_t, ytr_l, batch_size=BATCH_SIZE, shuffle=True, cache=True)
    val_ds   = make_dataset(Xva, yva_t, yva_l, batch_size=BATCH_SIZE, shuffle=False, cache=True)

    cb = [
        callbacks.EarlyStopping(monitor='val_T_battery_loss', patience=patience,
                                restore_best_weights=True, verbose=0, mode='min'),
        callbacks.ReduceLROnPlateau(monitor='val_T_battery_loss', factor=0.5,
                                    patience=patience//3, min_lr=1e-7, verbose=0, mode='min'),
        callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, 'tcn_gru_pretrain_ar04.keras'),
                                  monitor='val_T_battery_loss', save_best_only=True, verbose=0, mode='min'),
        ConciseEpochLogger(patience=patience, monitor='val_T_battery_loss'),
    ]

    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cb, verbose=0)
    tprint(f"  Pre-training completed: {len(hist.history['loss'])} epochs")
    del train_ds, val_ds, hist
    gc.collect()
    return model

def prepare_data(path, pcm_name, ar, global_scalers=None, for_global=False) -> DataBundle:
    df = clean(load_raw(path), f"{pcm_name} AR{ar}")
    df = engineer_features(df, ar, pcm_name)
    df = add_cumulative_features(df)
    df = add_lag_rolling(df)
    if for_global or global_scalers is not None:
        df = add_global_flags(df, pcm_name, ar)

    max_lag = max(LAG_STEPS)
    if len(df) <= max_lag + SEQ_STEPS + 50:
        raise ValueError(f"Dataset too short: {len(df)} rows")
    df = df.iloc[max_lag:].reset_index(drop=True).dropna()

    train_df, test_df = temporal_split(df, TRAIN_RATIO)
    T_initial = float(df['T_battery_clean'].iloc[0])

    train_max_time = train_df['time'].max() + 1e-6
    train_df['time_norm'] = train_df['time'] / train_max_time
    test_df['time_norm']  = test_df['time'] / train_max_time

    val_n  = int(len(train_df) * VAL_FRACTION)
    tr_xgb = train_df.iloc[:-val_n]
    va_xgb = train_df.iloc[-val_n:]

    xgb_cols = [c for c in get_xgb_cols() if c in train_df.columns]
    if for_global or global_scalers is not None:
        gf = ['is_rt45','is_lauric','is_palmitic','ar_0.3','ar_0.4','ar_0.5']
        xgb_cols += [c for c in gf if c in train_df.columns]

    X_tr_x = tr_xgb[xgb_cols].values;  y_tr_x = tr_xgb[XGB_TARGET].values
    X_va_x = va_xgb[xgb_cols].values;  y_va_x = va_xgb[XGB_TARGET].values
    X_te_x = test_df[xgb_cols].values; y_te_x = test_df[XGB_TARGET].values

    T_bat_lag3_va = va_xgb[TARGET_COL].shift(3).bfill().values
    T_bat_lag3_te = test_df['T_bat_lag3'].values

    tr_lstm = train_df.iloc[:-val_n].copy().reset_index(drop=True)
    va_lstm = train_df.iloc[-val_n:].copy().reset_index(drop=True)

    if global_scalers is not None:
        xs, ys_t_delta, ys_l, lstm_feats = global_scalers
        for c in lstm_feats:
            if c not in df.columns:
                df[c] = 0.0; train_df[c] = 0.0; test_df[c] = 0.0
                tr_lstm[c] = 0.0; va_lstm[c] = 0.0
    else:
        lstm_feats = list(FEATURE_COLS_CORE)
        if for_global:
            lstm_feats += ['is_rt45','is_lauric','is_palmitic','ar_0.3','ar_0.4','ar_0.5']
        xs = {c: RobustScaler().fit(tr_lstm[[c]]) for c in lstm_feats}
        ys_t_delta = RobustScaler().fit(tr_lstm[[XGB_TARGET]])
        ys_l = MinMaxScaler().fit(tr_lstm[['liquid_frac']])

    def scale(d):
        d2 = d.copy()
        for c in lstm_feats:
            d2[c] = xs[c].transform(d2[[c]])
        d2['dT_target_scaled'] = ys_t_delta.transform(d2[[XGB_TARGET]])
        d2['lf_target_scaled'] = ys_l.transform(d2[['liquid_frac']])
        return d2

    Xtr, ytr_t, ytr_l = build_sequences(scale(tr_lstm), lstm_feats, SEQ_STEPS,
                                        target_col='dT_target_scaled')
    Xva, yva_t, yva_l = build_sequences(scale(va_lstm), lstm_feats, SEQ_STEPS,
                                        target_col='dT_target_scaled')
    Xte, yte_t, yte_l = build_sequences(scale(test_df), lstm_feats, SEQ_STEPS,
                                        target_col='dT_target_scaled')

    return DataBundle(
        xgb={
            'Xtr': X_tr_x, 'ytr': y_tr_x,
            'Xva': X_va_x, 'yva': y_va_x,
            'Xte': X_te_x, 'yte': y_te_x,
            'cols': xgb_cols,
            'T_bat_lag3_va': T_bat_lag3_va,
            'T_bat_lag3_te': T_bat_lag3_te,
            'yva_abs': va_xgb[TARGET_COL].values,
            'yte_abs': test_df[TARGET_COL].values,
            'yva_abs_clean': va_xgb['T_battery_clean'].values,
            'yte_abs_clean': test_df['T_battery_clean'].values,
        },
        lstm={
            'Xtr':Xtr,'ytr_t':ytr_t,'ytr_l':ytr_l,
            'Xva':Xva,'yva_t':yva_t,'yva_l':yva_l,
            'Xte':Xte,'yte_t':yte_t,'yte_l':yte_l,
            'ys_t_delta':ys_t_delta,'ys_l':ys_l,'n_feat':len(lstm_feats),
            'val_true_t': va_lstm[TARGET_COL].values[SEQ_STEPS:],
            'val_true_t_clean': va_lstm['T_battery_clean'].values[SEQ_STEPS:],
            'T_prev_va': va_lstm['T_bat_lag3'].values[SEQ_STEPS:],
            'T_prev_te': test_df['T_bat_lag3'].values[SEQ_STEPS:],
        },
        test_df=test_df, train_df=train_df,
        pcm=pcm_name, ar=ar, T_initial=T_initial,
    )

def make_dataset(X, y_t, y_l, batch_size=BATCH_SIZE, shuffle=True, cache=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (X, {'T_battery': y_t, 'liquid_frac': y_l})
    )
    if cache:
        ds = ds.cache()
    if shuffle:
        ds = ds.shuffle(buffer_size=min(20000, len(X)), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# ─────────────────────────────────────────────────────────────────────────────

# METRICS

# ─────────────────────────────────────────────────────────────────────────────

In [5]:
def metrics(y_true_noisy, y_pred, y_persist=None, y_true_clean=None, liquid_frac=None):
    y_pred = np.asarray(y_pred).flatten()
    n = len(y_pred)

    yt_noisy = np.asarray(y_true_noisy).flatten()[:n]
    yt = np.asarray(y_true_clean).flatten()[:n] if y_true_clean is not None else yt_noisy

    err = np.abs(yt - y_pred)
    mae = float(np.mean(err))
    rmse = float(np.sqrt(np.mean((yt - y_pred)**2)))
    Tr = float(yt.max() - yt.min())
    Tm = float(yt.mean())

    r2 = float(r2_score(yt, y_pred)) if Tr > 1e-4 else float('nan')
    r2_noisy = float(r2_score(yt_noisy, y_pred)) if (yt_noisy.max() - yt_noisy.min()) > 1e-4 else float('nan')

    nrmse = rmse / (Tr + 1e-8)
    cvrmse = rmse / (abs(Tm) + 1e-8) * 100.0
    mape = float(np.mean(np.abs(err / (np.abs(yt) + 1e-6))) * 100)
    corr = float(stats.pearsonr(yt, y_pred)[0]) if Tr > 1e-4 else float('nan')
    acc = {f'acc_{t}K': float(np.mean(err <= t) * 100) for t in [0.5, 1.0, 2.0, 5.0, 10.0]}

    result = dict(
        mae=mae, rmse=rmse, r2=r2, r2_noisy=r2_noisy,
        nrmse=nrmse, cvrmse=cvrmse, mape=mape, pearson_r=corr,
        T_range=Tr, T_mean=Tm, n=n,
        rel_err_pct=mae / (abs(Tm) + 1e-6) * 100,
        pred_eff_pct=(1 - mae / (Tr + 1e-6)) * 100,
        **acc
    )

    if y_persist is not None:
        y_persist = np.asarray(y_persist).flatten()[:n]
        mae_persist = float(np.mean(np.abs(yt - y_persist)))
        result['mae_persist'] = mae_persist
        result['skill_vs_persist'] = float(1.0 - mae / (mae_persist + 1e-8))

    if liquid_frac is not None:
        lf = np.asarray(liquid_frac).flatten()[:n]
        zones = {
            'solid': lf <= 0.1,
            'melt': (lf > 0.1) & (lf < 0.9),
            'liquid': lf >= 0.9,
        }
        for zone_name, mask in zones.items():
            if mask.sum() > 5:
                result[f'n_{zone_name}'] = int(mask.sum())
                result[f'mae_{zone_name}'] = float(np.mean(np.abs(yt[mask] - y_pred[mask])))
                if y_persist is not None:
                    mae_p_zone = float(np.mean(np.abs(yt[mask] - y_persist[mask])))
                    result[f'skill_{zone_name}'] = float(1.0 - result[f'mae_{zone_name}'] / (mae_p_zone + 1e-8))
                if (yt[mask].max() - yt[mask].min()) > 1e-4:
                    result[f'r2_{zone_name}'] = float(r2_score(yt[mask], y_pred[mask]))

    # ── working-model R² (clean vs noisy) ─────────────────────────────────
    best_r2, r2_src, is_over = select_best_r2(result)
    result['best_r2']   = best_r2
    result['r2_src']    = r2_src
    result['is_overfit']= is_over
    # ─────────────────────────────────────────────────────────────────────

    return result

def _safe_r2_fmt(val):
    """Format R² safely handling None/NaN."""
    if val is None:
        return 'NaN'
    try:
        if np.isnan(val):
            return 'NaN'
    except TypeError:
        return 'NaN'
    return f"{val:.4f}"

def pmetrics(m, label=''):
    tag = f' [{label}]' if label else ''
    best_r2, r2_src, is_over = select_best_r2(m)

    if best_r2 is None:
        r2s = "NaN"
    else:
        r2s = f"{best_r2:.4f} ({r2_src})"

    tprint(f"    MAE (clean){tag:32}: {m['mae']:.6f} K")
    tprint(f"    Best R²{tag:36}: {r2s}")
    tprint(f"    RMSE{tag:39}: {m['rmse']:.6f} K")
    tprint(f"    NRMSE{tag:38}: {m['nrmse']:.6f}")
    prs = f"{m['pearson_r']:.4f}" if not np.isnan(m.get('pearson_r', float('nan'))) else 'NaN'
    tprint(f"    Pearson r{tag:35}: {prs}")
    tprint(f"    MAPE{tag:39}: {m['mape']:.4f}%")
    if 'skill_vs_persist' in m:
        tprint(f"    Skill vs Persist{tag:25}: {m['skill_vs_persist']:.4f}")
    if 'mae_melt' in m:
        tprint(f"    MAE melt-zone{tag:28}: {m['mae_melt']:.6f} K  (n={m.get('n_melt',0)})")
        if 'skill_melt' in m:
            tprint(f"    Skill melt-zone{tag:26}: {m['skill_melt']:.4f}")
    if 'mae_solid' in m:
        tprint(f"    MAE solid-zone{tag:27}: {m['mae_solid']:.6f} K  (n={m.get('n_solid',0)})")
    if 'mae_liquid' in m:
        tprint(f"    MAE liquid-zone{tag:26}: {m['mae_liquid']:.6f} K  (n={m.get('n_liquid',0)})")
    tprint(f"    Acc±5K{tag:37}: {m['acc_5.0K']:.1f}%  ±10K: {m['acc_10.0K']:.1f}%")

# ─────────────────────────────────────────────────────────────────────────────

# Physics Informed Moving Average (PIMA)

#### Note: This is not a neural network or a model, this is a purely statistical driven moving average function, that works an a 3s lagged temperature metric

# ─────────────────────────────────────────────────────────────────────────────

In [6]:
def build_pima_features(df, window=PIMA_WINDOW):
    d = df.copy()
    d['MA_T_lag']    = d['T_bat_lag3'].shift(1).rolling(window, min_periods=1).mean()
    d['Nu_x_lf']     = d['Nu'] * d['liquid_frac']
    d['Nu_x_solid']  = d['Nu'] * (1.0 - d['liquid_frac'])
    d['Nu_solid_dT'] = d['Nu_x_solid'] * d['dT_lag3']
    d = d.bfill().ffill()
    return d

def fit_pima(train_df, alpha=10.0):
    d = build_pima_features(train_df)
    feat_cols = ['MA_T_lag', 'Nu_x_lf', 'Nu_solid_dT', 'liquid_frac']
    X = d[feat_cols].values
    y = d['T_battery_clean'].values
    model = Ridge(alpha=alpha).fit(X, y)
    return model, feat_cols

def predict_pima(model, feat_cols, df, smooth_window=7):
    d = build_pima_features(df)
    X = d[feat_cols].values
    raw = model.predict(X)
    smoothed = pd.Series(raw).rolling(smooth_window, min_periods=1).mean().values
    return smoothed

# ─────────────────────────────────────────────────────────────────────────────

# MODEL — TCN + GRU & XGBOOST, ENSEMBLE

# ────────────────────────────────────────────────────────────────────────────

In [7]:
def _tcn_block(x, filters: int, kernel: int, dilation: int, l2: float = 5e-3):
    inp = x
    x = layers.Conv1D(filters, kernel, padding='causal', dilation_rate=dilation,
                      activation='linear', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.SpatialDropout1D(0.20)(x)
    x = layers.Conv1D(filters, kernel, padding='causal', dilation_rate=dilation,
                      activation='linear', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.SpatialDropout1D(0.20)(x)
    if inp.shape[-1] != filters:
        inp = layers.Conv1D(filters, 1, padding='same')(inp)
    return layers.Add()([x, inp])

def build_model(n_feat, seq_steps=SEQ_STEPS) -> keras.Model:
    inp = layers.Input(shape=(seq_steps, n_feat), name='input')
    x = layers.GaussianNoise(0.001)(inp)

    x = layers.Conv1D(48, 1, padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(5e-3))(x)
    x = layers.LayerNormalization()(x)

    for dil in [1, 2, 4, 8]:
        x = _tcn_block(x, filters=48, kernel=3, dilation=dil, l2=5e-3)

    x = layers.GRU(48, return_sequences=True, dropout=0.20, recurrent_dropout=0.10,
                   kernel_regularizer=regularizers.l2(5e-3))(x)
    x = layers.LayerNormalization()(x)
    h = layers.GRU(24, return_sequences=False, dropout=0.20, recurrent_dropout=0.10,
                   kernel_regularizer=regularizers.l2(5e-3))(x)
    h = layers.LayerNormalization()(h)
    h = layers.Dropout(0.20)(h)

    sh = layers.Dense(48, activation='swish',
                      kernel_regularizer=regularizers.l2(5e-3))(h)
    sh = layers.Dropout(0.15)(sh)
    sh = layers.Dense(24, activation='swish',
                      kernel_regularizer=regularizers.l2(5e-3))(sh)

    out_t  = layers.Dense(1, activation='linear',  dtype='float32', name='T_battery')(sh)
    out_lf = layers.Dense(1, activation='sigmoid', dtype='float32', name='liquid_frac')(sh)

    model = keras.Model(inputs=inp, outputs=[out_t, out_lf], name='PCM_TCN_GRU_v2_3')
    model.compile(
        optimizer=optimizers.Adam(2e-4, clipnorm=1.0),
        loss={'T_battery': 'huber', 'liquid_frac': 'mse'},
        loss_weights={'T_battery': 1.0, 'liquid_frac': 0.15},
        metrics={'T_battery': ['mae'], 'liquid_frac': ['mae']},
    )
    return model

# ── ADDED: Concise epoch-level logger (replaces bulky batch logs) ────────────
class ConciseEpochLogger(callbacks.Callback):
    def __init__(self, patience=60, monitor='val_T_battery_loss'):
        super().__init__()
        self.patience = patience
        self.monitor = monitor
        self.wait = 0
        self.best = float('inf')
        self._prev_lr = None

    def on_train_begin(self, logs=None):
        self.epochs = self.params['epochs']
        tprint(f"  🚀 Training {self.epochs} epochs | monitor={self.monitor}")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor, logs.get('val_loss', float('inf')))
        val_mae = logs.get('val_T_battery_mae', logs.get('val_mae', float('nan')))
        lr = logs.get('learning_rate', float('nan'))
        if np.isnan(lr):
            try:
                lr = float(self.model.optimizer.learning_rate.numpy())
            except Exception:
                lr = float('nan')

        prog = int(20 * (epoch + 1) / self.epochs)
        bar = '█' * prog + '░' * (20 - prog)

        if current < self.best - 1e-6:
            self.best = current
            self.wait = 0
            status = '\033[92m✓\033[0m'
        else:
            self.wait += 1
            if self.wait >= self.patience:
                status = '\033[91m⏹ STOP\033[0m'
            elif self.wait >= self.patience * 2 // 3:
                status = '\033[93m⚠\033[0m'
            else:
                status = '\033[90m-\033[0m'

        tprint(f"  [{bar}] \033[96mEpoch {epoch+1:03d}/{self.epochs}\033[0m | "
               f"val_loss={current:.4f} val_mae={val_mae:.4f} lr={lr:.2e} {status}")

        if self._prev_lr is not None and lr < self._prev_lr * 0.9:
            tprint(f"  \033[93m↻ LR dropped: {self._prev_lr:.2e} → {lr:.2e}\033[0m")
        self._prev_lr = lr

    def on_train_end(self, logs=None):
        final_epoch = len(self.model.history.history.get('loss', []))
        if final_epoch < self.epochs:
            tprint(f"  \033[91m⏹ Early stop at epoch {final_epoch}/{self.epochs} (best restored)\033[0m")
        else:
            tprint(f"  \033[92m✓ Finished all {self.epochs} epochs\033[0m")

# ── MODIFIED: callbacks now include ConciseEpochLogger, internal verbose=0 ─
def get_callbacks(name: str, patience: int = PATIENCE):
    return [
        callbacks.EarlyStopping(
            monitor='val_T_battery_loss', patience=patience,
            restore_best_weights=True, verbose=0, mode='min',
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_T_battery_loss', factor=0.5,
            patience=patience // 3,
            min_lr=1e-7, verbose=0, mode='min', min_delta=1e-5,
        ),
        callbacks.ModelCheckpoint(
            os.path.join(OUTPUT_DIR, f'{name}_best.keras'),
            monitor='val_T_battery_loss', save_best_only=True,
            verbose=0, mode='min',
        ),
        ConciseEpochLogger(patience=patience, monitor='val_T_battery_loss'),
    ]
def _make_xgb_regressor(early_stopping_rounds=80) -> xgb.XGBRegressor:
    params = dict(XGB_PARAMS)
    if XGB_VERSION >= (1, 6):
        params['callbacks'] = [xgb.callback.EarlyStopping(
            rounds=early_stopping_rounds, metric_name='rmse',
            data_name='validation_0', save_best=True,
        )]
        return xgb.XGBRegressor(**params)
    else:
        _rounds = early_stopping_rounds
        class _LegacyXGB(xgb.XGBRegressor):
            def fit(self, X, y, eval_set=None, verbose=False, **kw):
                return super().fit(X, y, eval_set=eval_set,
                                   early_stopping_rounds=_rounds,
                                   verbose=verbose, **kw)
        return _LegacyXGB(**params)
        
def ensemble_standard(xp, lp, pp, yt, xvp, lvp, pvp, yv):
    xv_rmse = float(np.sqrt(np.mean((yv - xvp)**2)))
    lv_rmse = float(np.sqrt(np.mean((yv - lvp)**2)))
    pv_rmse = float(np.sqrt(np.mean((yv - pvp)**2)))
    xv_mae  = float(np.mean(np.abs(yv - xvp)))
    lv_mae  = float(np.mean(np.abs(yv - lvp)))
    pv_mae  = float(np.mean(np.abs(yv - pvp)))

    tprint(f"    [Ens-Std] XGB  val MAE={xv_mae:.4f}  RMSE={xv_rmse:.4f}")
    tprint(f"    [Ens-Std] LSTM val MAE={lv_mae:.4f}  RMSE={lv_rmse:.4f}")
    tprint(f"    [Ens-Std] PIMA val MAE={pv_mae:.4f}  RMSE={pv_rmse:.4f}")

    best_w    = {'xgb': 1.0/3.0, 'lstm': 1.0/3.0, 'pima': 1.0/3.0, 'bias': 0.0}
    best_rmse = float('inf')

    for wx in np.arange(0.0, 1.01, 0.05):
        for wl in np.arange(0.0, 1.01 - wx, 0.05):
            wp = round(1.0 - wx - wl, 6)
            if wp < -1e-9:
                continue
            wp = max(wp, 0.0)
            ev   = wx*xvp + wl*lvp + wp*pvp
            rmse = float(np.sqrt(np.mean((yv - ev)**2)))
            if rmse < best_rmse:
                best_rmse = rmse
                best_w = {'xgb': wx, 'lstm': wl, 'pima': wp, 'bias': 0.0}

    ep      = best_w['xgb']*xp + best_w['lstm']*lp + best_w['pima']*pp
    ev_pred = best_w['xgb']*xvp + best_w['lstm']*lvp + best_w['pima']*pvp

    if FALLBACK_TO_BEST_SINGLE:
        ev_rmse        = float(np.sqrt(np.mean((yv - ev_pred)**2)))
        best_single    = min(xv_rmse, lv_rmse, pv_rmse)
        if ev_rmse > best_single * 1.40:
            if xv_rmse == best_single:
                ep = xp.copy(); best_w = {'xgb':1.,'lstm':0.,'pima':0.,'bias':0.}
                tprint("    [Ens-Std] Fallback → XGB")
            elif lv_rmse == best_single:
                ep = lp.copy(); best_w = {'xgb':0.,'lstm':1.,'pima':0.,'bias':0.}
                tprint("    [Ens-Std] Fallback → TCN-GRU")
            else:
                ep = pp.copy(); best_w = {'xgb':0.,'lstm':0.,'pima':1.,'bias':0.}
                tprint("    [Ens-Std] Fallback → PIMA")

    return ep, best_w

def ensemble_ridge(xp, lp, pp, xvp, lvp, pvp, yv):
    """
    Ridge ensemble with hard non-negativity guard.
    When base-model error scales differ by >10x (PIMA vs TCN-GRU),
    unconstrained Ridge extrapolates and produces MAE > 1 K.
    Fix: fit without intercept on non-negative inputs; fall back to
    TCN-GRU-only if any weight is pathological.
    """
    X_val = np.vstack([xvp, lvp, pvp]).T
    y_val = yv.flatten()

    # Fit without intercept; alpha=10 for stability
    ridge = Ridge(alpha=10.0, fit_intercept=False).fit(X_val, y_val)
    coef  = ridge.coef_.flatten()

    # Pathology check: negative weights or near-zero sum → use TCN-GRU only
    if np.any(coef < -1e-3) or coef.sum() < 0.05:
        tprint("    [Ens-Ridge] Pathological weights detected → fallback to TCN-GRU only")
        coef = np.array([0.0, 1.0, 0.0])

    # Normalise so weights sum to 1 (prevents scale drift)
    coef = np.clip(coef, 0.0, None)
    if coef.sum() > 0:
        coef = coef / coef.sum()
    else:
        coef = np.array([0.0, 1.0, 0.0])

    X_test = np.vstack([xp, lp, pp]).T
    ep = X_test @ coef

    w = {'xgb':  float(coef[0]),
         'lstm': float(coef[1]),
         'pima': float(coef[2]),
         'bias': 0.0}
    tprint(f"    [Ens-Ridge] Weights: XGB={w['xgb']:.3f} TCN-GRU={w['lstm']:.3f} "
           f"PIMA={w['pima']:.3f} bias={w['bias']:.3f}")
    return ep, w

def ensemble_weighted_median(xp, lp, pp, xvp, lvp, pvp, yv):
    xv_rmse = float(np.sqrt(np.mean((yv - xvp)**2))) + 1e-8
    lv_rmse = float(np.sqrt(np.mean((yv - lvp)**2))) + 1e-8
    pv_rmse = float(np.sqrt(np.mean((yv - pvp)**2))) + 1e-8
    inv_sum = 1/xv_rmse + 1/lv_rmse + 1/pv_rmse
    wx = (1/xv_rmse) / inv_sum
    wl = (1/lv_rmse) / inv_sum
    wp = (1/pv_rmse) / inv_sum

    nx = max(1, round(wx * 10))
    nl = max(1, round(wl * 10))
    np_ = max(1, round(wp * 10))
    stack = np.vstack([np.tile(xp, (nx,1)),
                       np.tile(lp, (nl,1)),
                       np.tile(pp, (np_,1))])
    ep = np.median(stack, axis=0)
    w  = {'xgb': wx, 'lstm': wl, 'pima': wp, 'bias': 0.0}
    tprint(f"    [Ens-Med]   Weights: XGB={wx:.3f} TCN-GRU={wl:.3f} PIMA={wp:.3f}")
    return ep, w

# ─────────────────────────────────────────────────────────────────────────────

# PER-AR PIPELINE

# ─────────────────────────────────────────────────────────────────────────────

In [8]:
def run_ar(ar, path, pcm_name, global_scalers=None, pretrain_model_path=None):
    name = f'{pcm_name}_AR{ar}'
    tprint(f'\n{"="*60}\n  {name}\n{"="*60}')

    data = prepare_data(path, pcm_name, ar, global_scalers=global_scalers, for_global=True)
    d    = data
    T0   = data.T_initial

    # ── XGBoost ─────────────────────────────────────────────────────────────
    tprint(f'[{name}] XGBoost (delta target) ...')
    xm = _make_xgb_regressor(early_stopping_rounds=80)
    xm.fit(d.xgb['Xtr'], d.xgb['ytr'],
           eval_set=[(d.xgb['Xva'], d.xgb['yva'])],
           verbose=False)

    delta_te = xm.predict(d.xgb['Xte'])
    xp       = d.xgb['T_bat_lag3_te'] + delta_te

    delta_va = xm.predict(d.xgb['Xva'])
    xm_val   = d.xgb['T_bat_lag3_va'] + delta_va

    persist_te = d.xgb['T_bat_lag3_te']
    xm_met = metrics(
        d.xgb['yte_abs'], xp,
        y_persist=persist_te,
        y_true_clean=d.xgb['yte_abs_clean'],
        liquid_frac=data.test_df['liquid_frac'].values[:len(xp)]
    )
    pmetrics(xm_met, f'XGB {name}')

    # ── TCN-GRU ─────────────────────────────────────────────────────────────
    tprint(f'[{name}] TCN-GRU (curriculum / delta-T) ...')
    tf.keras.backend.clear_session()
    lm = build_model(d.lstm['n_feat'], SEQ_STEPS)
    tprint(f"  Model params: {lm.count_params():,}")

    fine_tune_epochs = EPOCHS
    fine_tune_patience = PATIENCE
    if pretrain_model_path and os.path.exists(pretrain_model_path):
        tprint(f"  Loading pre-trained weights: {pretrain_model_path}")
        lm_pre = keras.models.load_model(pretrain_model_path)
        lm.set_weights(lm_pre.get_weights())
        del lm_pre
        gc.collect()
        lm.compile(
            optimizer=optimizers.Adam(5e-5, clipnorm=1.0),
            loss={'T_battery': 'huber', 'liquid_frac': 'mse'},
            loss_weights={'T_battery': 1.0, 'liquid_frac': 0.15},
            metrics={'T_battery': ['mae'], 'liquid_frac': ['mae']},
        )
        fine_tune_epochs = 150
        fine_tune_patience = 25
        tprint(f"  Fine-tuning: epochs={fine_tune_epochs}, patience={fine_tune_patience}, LR=5e-5")
    else:
        tprint("  No pre-trained model found; training from scratch.")

    train_ds = make_dataset(d.lstm['Xtr'], d.lstm['ytr_t'], d.lstm['ytr_l'],
                            batch_size=BATCH_SIZE, shuffle=True, cache=True)
    val_ds   = make_dataset(d.lstm['Xva'], d.lstm['yva_t'], d.lstm['yva_l'],
                            batch_size=BATCH_SIZE, shuffle=False, cache=True)

    hist = None
    try:
        # MODIFIED: verbose=0, custom ConciseEpochLogger handles output
        hist = lm.fit(train_ds, validation_data=val_ds, epochs=fine_tune_epochs,
                      callbacks=get_callbacks(name, patience=fine_tune_patience), verbose=0)
        tprint(f"  Trained {len(hist.history['loss'])} epochs")
    except BaseException as fit_err:
        tprint(f"  [WARN] TCN-GRU fit interrupted/raised: {fit_err}. Using last-epoch weights.")
    finally:
        del train_ds, val_ds
        gc.collect()

    best_model_path = os.path.join(OUTPUT_DIR, f'{name}_best.keras')
    if os.path.exists(best_model_path):
        lm = keras.models.load_model(best_model_path)
    else:
        tprint(f"  [WARN] Best model checkpoint not found; using last-epoch weights.")

    pd_t, pd_l = lm.predict(d.lstm['Xte'], batch_size=BATCH_SIZE, verbose=0)
    delta_pred = d.lstm['ys_t_delta'].inverse_transform(pd_t.reshape(-1,1)).flatten()
    T_prev_te  = d.lstm['T_prev_te'][:len(delta_pred)]
    lp         = T_prev_te + delta_pred
    lp_lf      = d.lstm['ys_l'].inverse_transform(pd_l.reshape(-1,1)).flatten()

    lt       = data.test_df['T_battery'].values[SEQ_STEPS : SEQ_STEPS + len(lp)]
    lt_clean = data.test_df['T_battery_clean'].values[SEQ_STEPS : SEQ_STEPS + len(lp)]
    lt_lf    = data.test_df['liquid_frac'].values[SEQ_STEPS : SEQ_STEPS + len(lp_lf)]

    persist_lstm_te = d.lstm['T_prev_te'][:len(lt)]
    lm_met = metrics(
        lt, lp,
        y_persist=persist_lstm_te,
        y_true_clean=lt_clean,
        liquid_frac=lt_lf
    )
    pmetrics(lm_met, f'TCN-GRU {name}')

    # ── PIMA ──────────────────────────────────────────────────────────────
    tprint(f'[{name}] PIMA (physics MA, {PIMA_WINDOW}s window) ...')
    pima_model, pima_cols = fit_pima(data.train_df, alpha=10.0)

    pima_raw  = predict_pima(pima_model, pima_cols, data.test_df, smooth_window=7)
    pima_pred = pima_raw[SEQ_STEPS : SEQ_STEPS + len(lp)]

    val_n       = int(len(data.train_df) * VAL_FRACTION)
    pima_va_df  = data.train_df.iloc[-val_n:].copy().reset_index(drop=True)
    pima_va_raw = predict_pima(pima_model, pima_cols, pima_va_df, smooth_window=7)

    n_vt    = len(d.lstm['val_true_t'])
    pima_va = pima_va_raw[SEQ_STEPS : SEQ_STEPS + n_vt]

    pima_met = metrics(
        lt[:len(pima_pred)], pima_pred[:len(lt)],
        y_persist=d.lstm['T_prev_te'][:len(pima_pred)],
        y_true_clean=lt_clean[:len(pima_pred)],
        liquid_frac=lt_lf[:len(pima_pred)]
    )
    pmetrics(pima_met, f'PIMA {name}')

    coeffs = dict(zip(pima_cols, pima_model.coef_))
    tprint(f"    PIMA eq: T_pred = {coeffs['MA_T_lag']:.4f}·MA_T_lag + "
           f"{coeffs['Nu_x_lf']:.6f}·Nu·lf + {coeffs['Nu_solid_dT']:.6f}·Nu·(1-lf)·dT + "
           f"{coeffs['liquid_frac']:.4f}·lf + {pima_model.intercept_:.4f}")

    # ── VALIDATION alignment ───────────────────────────────────────────────
    vp_t, _ = lm.predict(d.lstm['Xva'], batch_size=BATCH_SIZE, verbose=0)
    delta_va_pred = d.lstm['ys_t_delta'].inverse_transform(vp_t.reshape(-1,1)).flatten()
    T_prev_va     = d.lstm['T_prev_va'][:len(delta_va_pred)]
    lv_K          = T_prev_va + delta_va_pred
    xv_K          = xm_val[SEQ_STEPS : SEQ_STEPS + n_vt]
    yv_K          = d.lstm['val_true_t_clean'][:n_vt]

    nv = min(len(xv_K), len(lv_K), len(pima_va), len(yv_K))
    xv_K, lv_K, pima_va, yv_K = xv_K[:nv], lv_K[:nv], pima_va[:nv], yv_K[:nv]

    if hist is not None:
        hist_df = pd.DataFrame(hist.history)
        hist_df.to_csv(os.path.join(OUTPUT_DIR, f'{name}_history.csv'), index=False)
    else:
        hist_df = pd.DataFrame()

    xm.save_model(os.path.join(OUTPUT_DIR, f'{name}_xgb.json'))
    del lm; keras.backend.clear_session(); gc.collect()

    # ── Trim ───────────────────────────────────────────────────────────────
    n   = len(lp)
    xpa = xp[SEQ_STEPS : SEQ_STEPS + n]
    ppa = pima_pred[:n]

    mn  = min(len(xpa), n, len(ppa), len(lt))
    xpa, lp, ppa, lt, lt_clean, lt_lf, lp_lf = (
        xpa[:mn], lp[:mn], ppa[:mn], lt[:mn], lt_clean[:mn], lt_lf[:mn], lp_lf[:mn]
    )
    if mn < n:
        tprint(f"  [WARN] Trimmed to {mn} samples for alignment")

    lf_eval = lt_lf[:mn]

    # ── Ensembles ──────────────────────────────────────────────────────────
    ep_std, w_std = ensemble_standard(xpa, lp, ppa, lt, xv_K, lv_K, pima_va, yv_K)
    em_std = metrics(lt[:mn], ep_std, y_persist=d.lstm['T_prev_te'][:mn],
                     y_true_clean=lt_clean[:mn], liquid_frac=lf_eval)
    tprint(f"\n  [ENSEMBLE STANDARD] {name}")
    pmetrics(em_std, f'ENS-Std {name}')
    tprint(f"  Weights: XGB={w_std['xgb']:.3f} TCN-GRU={w_std['lstm']:.3f} "
           f"PIMA={w_std['pima']:.3f} bias={w_std['bias']:.4f}")

    ep_ridge, w_ridge = ensemble_ridge(xpa, lp, ppa, xv_K, lv_K, pima_va, yv_K)
    em_ridge = metrics(lt[:mn], ep_ridge, y_persist=d.lstm['T_prev_te'][:mn],
                       y_true_clean=lt_clean[:mn], liquid_frac=lf_eval)
    tprint(f"\n  [ENSEMBLE RIDGE] {name}")
    pmetrics(em_ridge, f'ENS-Ridge {name}')

    ep_med, w_med = ensemble_weighted_median(xpa, lp, ppa, xv_K, lv_K, pima_va, yv_K)
    em_med = metrics(lt[:mn], ep_med, y_persist=d.lstm['T_prev_te'][:mn],
                     y_true_clean=lt_clean[:mn], liquid_frac=lf_eval)
    tprint(f"\n  [ENSEMBLE MEDIAN] {name}")
    pmetrics(em_med, f'ENS-Med {name}')

    best_ens_pred    = ep_std
    best_ens_metrics = em_std
    best_ens_weights = w_std
    for ep_cand, em_cand, wc, tag in [
        (ep_med, em_med, w_med, 'Median'),
    ]:
        cand_r2 = em_cand.get('best_r2', em_cand.get('r2', float('nan')))
        if cand_r2 is None:
            cand_r2 = float('nan')
        best_r2 = best_ens_metrics.get('best_r2', best_ens_metrics.get('r2', float('nan')))
        if best_r2 is None:
            best_r2 = float('nan')
        if cand_r2 > best_r2:
            best_ens_pred    = ep_cand
            best_ens_metrics = em_cand
            best_ens_weights = wc
            tprint(f"  [INFO] Best ensemble switched to {tag} "
                   f"(R²={cand_r2:.4f})")

    return {
        'pcm': pcm_name, 'ar': ar, 'name': name,
        'y_true': lt_clean, 'y_true_noisy': lt, 'y_true_lf': lt_lf,
        'xgb_pred': xpa, 'lstm_pred': lp, 'pima_pred': ppa,
        'lstm_pred_lf': lp_lf,
        'ens_pred':          best_ens_pred,
        'ens_metrics':       best_ens_metrics,
        'weights':           best_ens_weights,
        'ens_std_pred':      ep_std,
        'ens_std_metrics':   em_std,
        'ens_ridge_pred':    ep_ridge,
        'ens_ridge_metrics': em_ridge,
        'ens_med_pred':      ep_med,
        'ens_med_metrics':   em_med,
        'xgb_metrics':  xm_met,
        'lstm_metrics': lm_met,
        'pima_metrics': pima_met,
        'xgb_model': xm, 'xgb_cols': d.xgb['cols'],
        'T_initial': T0, 'test_df': data.test_df, 'hist_df': hist_df,
    }

# ─────────────────────────────────────────────────────────────────────────────

# NCA & E-CHEM VALIDATION, PHYSICS-BASED AR & PCM SELECTION

# ─────────────────────────────────────────────────────────────────────────────

In [9]:
def prepare_nca(df):
    rename = {
        "Test_Time(s)":"TestTimes","Step_Time(s)":"StepTimes",
        "Step_Index":"StepIndex","Voltage(V)":"VoltageV",
        "Current(A)":"CurrentA","Surface_Temp(degC)":"SurfaceTempdegC",
    }
    d = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
    if "Date_Time" in d.columns:
        d["Date_Time"] = pd.to_datetime(d["Date_Time"], errors="coerce")
    d = d.sort_values("TestTimes").reset_index(drop=True)
    d["TsurfaceK"]  = d["SurfaceTempdegC"] + 273.15
    d["deltaTK"]    = d["TsurfaceK"] - T_AMBIENT_K
    d["SoCproxy"]   = (d["VoltageV"] - 2.5) / (4.2 - 2.5)
    d["powerW"]     = d["VoltageV"] * d["CurrentA"].abs()
    d["dVdt"]       = d["VoltageV"].diff().fillna(0.0)
    d["dTdt"]       = d["SurfaceTempdegC"].diff().fillna(0.0)
    d["rollmean_T"] = d["SurfaceTempdegC"].rolling(5, min_periods=1).mean()
    d["cummax_T"]   = d["SurfaceTempdegC"].cummax()
    d["T_rise"]     = d["SurfaceTempdegC"] - d["SurfaceTempdegC"].iloc[0]
    return d

def load_nca():
    frames = []
    for key, path in NCA_DATA_PATHS.items():
        if not os.path.exists(path):
            tprint(f"  [SKIP] NCA: {path}"); continue
        raw = pd.read_excel(path); raw["run"] = key
        frames.append(prepare_nca(raw))
    if not frames: return pd.DataFrame()
    nca       = pd.concat(frames, ignore_index=True)
    discharge = nca[nca["StepIndex"] == NCA_DISCHARGE_STEP].copy()
    discharge = discharge.sort_values(["run","TestTimes"]).reset_index(drop=True)
    tprint(f"  NCA: {len(discharge)} rows, {discharge['run'].nunique()} runs")
    return discharge

def load_specs():
    if not os.path.exists(MANUFACTURER_SPEC_PATH):
        tprint("  [SKIP] Manufacturer specs not found"); return pd.DataFrame()
    df = pd.read_excel(MANUFACTURER_SPEC_PATH)
    tprint("\n  Manufacturer Specifications:\n" + df.to_string(index=False))
    return df

def pcm_safety_check(nca_df):
    max_t  = float(nca_df["SurfaceTempdegC"].max())
    heat_j = float(nca_df["powerW"].sum())
    cap_j  = PCM_LATENT_HEAT_MEAN * PCM_MASS_KG
    return {
        "max_surface_temp_C":    max_t,
        "peak_heat_input_J":     heat_j,
        "pcm_latent_capacity_J": cap_j,
        "pcm_sufficient":        bool(cap_j >= heat_j),
        "comment": "PCM adequate" if cap_j >= heat_j else "WARNING: PCM latent heat insufficient",
    }

def select_best_ar_physics(results, pcm_name):
    scores = {}
    for ar in ['0.3', '0.4', '0.5']:
        key = f"{pcm_name}_AR{ar}"
        if key not in results:
            continue
        res = results[key]
        df = res['test_df']

        # ── FIX 1: True baseline from experiment start, not test-set start ──
        T0 = res['T_initial']
        T_bat = (df['T_battery_clean'].values
                 if 'T_battery_clean' in df.columns
                 else df['T_battery'].values)
        peak_T = float(T_bat.max())
        rise_K = max(0.0, peak_T - T0)

        # ── FIX 2: Model-trust gate (reject isothermal / flat-line test sets) ──
        ens_r2 = res['ens_metrics'].get('best_r2', res['ens_metrics'].get('r2', 0.0))
        if ens_r2 is None or np.isnan(ens_r2):
            ens_r2 = 0.0
        test_var = float(np.var(T_bat))
        # A flat test set (variance < 0.5 K²) means the model is not being tested
        # on thermal transients, so metrics are meaningless.
        is_isothermal = test_var < 0.5
        model_trust = 0.05 if (is_isothermal or ens_r2 < 0.85) else 1.0

        # ── FIX 3: Plateau & dynamics with proper normalization ──
        plateau_mask = (df['liquid_frac'] > 0.1) & (df['liquid_frac'] < 0.9)
        plateau_dur = int(plateau_mask.sum())
        # Normalize against a realistic maximum (~8000 steps) instead of clipping at 1000
        plateau_score = min(plateau_dur / 8000.0, 1.0)

        if plateau_mask.sum() > 10:
            dTdt = np.abs(np.gradient(T_bat[plateau_mask])).mean()
            T_std = np.std(T_bat[plateau_mask])
        else:
            skip = max(1, len(T_bat) // 20)
            dTdt = np.abs(np.gradient(T_bat[skip:])).mean()
            T_std = np.std(T_bat[skip:])

        # Tighter thresholds to actually distinguish good from poor regulation
        flatness_score = float(np.exp(-dTdt / 0.01))
        uniformity_score = float(np.exp(-T_std / 2.0))

        # ── FIX 4: Theoretical optimum AR = 0.4 for cylindrical PCM wraps ──
        ar_val = float(ar)
        ar_optimal = 0.4
        ar_score = float(np.exp(-((ar_val - ar_optimal)**2) / 0.02))

        # ── FIX 5: Battery-activity-aware thermal performance ──
        # Allowable rise: 25 K (25 °C ambient → 50 °C battery limit)
        rise_allowable = 25.0
        rise_score = float(np.exp(-rise_K / rise_allowable))

        # Composite: peak temp (30%), plateau (25%), flatness (20%),
        #            theoretical AR (15%), uniformity (10%)
        score = (
            0.30 * rise_score +
            0.25 * plateau_score +
            0.20 * flatness_score +
            0.15 * ar_score +
            0.10 * uniformity_score
        ) * model_trust

        scores[ar] = {
            'composite': float(score),
            'plateau_steps': plateau_dur,
            'peak_rise_K': rise_K,
            'peak_T': peak_T,
            'dTdt_melting': dTdt,
            'T_std_melting': T_std,
            'rise_score': rise_score,
            'plateau_score': plateau_score,
            'flatness_score': flatness_score,
            'uniformity_score': uniformity_score,
            'ar_score': ar_score,
            'model_trust': model_trust,
            'ens_r2': ens_r2,
            'is_isothermal': is_isothermal,
        }
    if not scores:
        return None, {}
    best = max(scores, key=lambda k: scores[k]['composite'])
    return best, scores


def select_best_pcm_overall(results):
    winners = {}
    for pcm_name in PCM_REGISTRY:
        best_ar, scores = select_best_ar_physics(results, pcm_name)
        if best_ar:
            key = f"{pcm_name}_AR{best_ar}"
            if key in results:
                winners[pcm_name] = {
                    'ar': best_ar,
                    'res': results[key],
                    'scores': scores[best_ar],
                }

    if not winners:
        return None, {}

    # ── PCM Material Property Scoring (battery-activity-aware) ──
    # For 5C NCA discharge (high battery activity), the PCM must:
    #   1. Melt before the battery hits ~50 °C (≈ 323 K)
    #   2. Have high latent heat to absorb the thermal spike
    #   3. Have a melting range wide enough to buffer transients
    T_optimal_center = 318.15   # 45 °C
    T_max_safe = 323.15          # 50 °C (hard limit for Li-ion)
    max_L = 210_000.0
    max_k = 0.25

    for pcm_name, info in winners.items():
        props = PCM_PROPS[pcm_name]

        # 1. Latent heat (higher = better absorption of battery heat)
        latent_score = props['L_latent'] / max_L

        # 2. Melting temperature (must be below battery safety limit)
        T_center = (props['T_liquidus'] + props['T_solidus']) / 2.0
        if T_center > T_max_safe:
            # Palmitic Acid melts at ~60 °C: battery is already in thermal runaway
            # before PCM even activates. Heavy penalty.
            temp_score = 0.05
        else:
            temp_score = float(np.exp(-((T_center - T_optimal_center)**2) / 50.0))

        # 3. Melting range (optimal ~5 K; too narrow = sudden, too wide = diffuse)
        T_range = props['T_liquidus'] - props['T_solidus']
        range_score = float(np.exp(-((T_range - 5.0)**2) / 30.0))

        # 4. Thermal conductivity (liquid, higher = better heat transfer)
        k_score = props['k_liquid'] / max_k

        # 5. Latent capacity vs battery heat load (5C discharge)
        #    Typical peak heat input from NCA data ≈ 18.8 kJ
        pcm_capacity_J = props['L_latent'] * PCM_MASS_KG
        capacity_score = min(pcm_capacity_J / 20_000.0, 1.0)

        # Material composite: latent (30%), melting temp (30%), range (20%),
        #                     conductivity (10%), capacity (10%)
        material_score = (
            0.30 * latent_score +
            0.30 * temp_score +
            0.20 * range_score +
            0.10 * k_score +
            0.10 * capacity_score
        )

        info['material_score'] = material_score
        info['latent_score'] = latent_score
        info['temp_score'] = temp_score
        info['range_score'] = range_score
        info['capacity_score'] = capacity_score

    # ── Overall Winner: 60% material science, 40% AR thermal performance ──
    # This prevents an isothermal test-set artifact from overriding physics.
    best_pcm = None
    best_total = -1.0

    for pcm_name, info in winners.items():
        ar_perf = info['scores']['composite']
        mat_score = info['material_score']
        total = 0.60 * mat_score + 0.40 * ar_perf

        s = info['scores']
        tprint(f"  [{pcm_name} AR{info['ar']}] "
               f"AR_perf={ar_perf:.3f}  material={mat_score:.3f}  "
               f"TOTAL={total:.3f}  |  "
               f"plateau={s['plateau_steps']}  rise={s['peak_rise_K']:.2f}K  "
               f"trust={s['model_trust']:.1f}  R²={s['ens_r2']:.3f}")

        if total > best_total:
            best_total = total
            best_pcm = pcm_name

    return best_pcm, winners

# ─────────────────────────────────────────────────────────────────────────────

# VISUALISATIONS , MAIN CODE

# ─────────────────────────────────────────────────────────────────────────────

In [10]:
def _save(fig, fname):
    p = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(p, dpi=300, bbox_inches='tight')
    plt.close(fig)
    tprint(f"  Saved: {p}")

def plot_training_curves(hist_df, name):
    if hist_df is None or hist_df.empty:
        tprint(f"  [SKIP] No history to plot for {name}")
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, (tc, vc), title in zip(axes,
        [('T_battery_loss',    'val_T_battery_loss'),
         ('liquid_frac_loss',  'val_liquid_frac_loss')],
        ['T_battery Loss (Huber)', 'liquid_frac Loss (MSE)']):
        if tc in hist_df.columns: ax.plot(hist_df[tc], label='Train', color='steelblue')
        if vc in hist_df.columns: ax.plot(hist_df[vc], label='Val',   color='tomato')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f'Training Curves — {name}', fontweight='bold')
    plt.tight_layout(); _save(fig, f'train_curves_{name}.png')

def plot_prediction_vs_actual(res, name):
    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    t  = np.arange(len(res['y_true']))
    ax = axes[0]
    ax.plot(t, res['y_true'],        'k-',  lw=2.0,  label='Actual T_battery (clean)', zorder=5)
    ax.plot(t, res['xgb_pred'],      color='#4e8ac8', lw=1.2, ls='--', label='XGB',         alpha=0.85)
    ax.plot(t, res['lstm_pred'],     color='#e07b39', lw=1.2, ls='--', label='TCN-GRU',      alpha=0.85)
    ax.plot(t, res['pima_pred'],     color='#9b59b6', lw=1.2, ls='-.', label='PIMA',         alpha=0.85)
    ax.plot(t, res['ens_pred'],      color='#c0392b', lw=2.0,           label='Best Ensemble',zorder=4)
    ax.plot(t, res['ens_std_pred'],  color='#e67e22', lw=1.2, ls=':',  label='Ens-Std',      zorder=3)
    ax.plot(t, res['ens_ridge_pred'],color='#2ecc71', lw=1.2, ls=':',  label='Ens-Ridge',    zorder=3)
    m = res['ens_metrics']
    r2_val = m.get('best_r2', m.get('r2', float('nan')))
    r2s = _safe_r2_fmt(r2_val)
    prs = f"{m.get('pearson_r', float('nan')):.4f}" if not np.isnan(m.get('pearson_r', float('nan'))) else 'NaN'
    ax.set_title(f"{name} — T_battery | MAE={m['mae']:.4f} K  R²={r2s}  r={prs}", fontweight='bold')
    ax.set_ylabel('T_battery (K)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax = axes[1]
    ax.plot(t, res['y_true_lf'],   'k-',  lw=2,  label='Actual lf',   zorder=5)
    ax.plot(t, res['lstm_pred_lf'],color='#27ae60', lw=1.8, ls='--', label='Predicted lf')
    ax.set_ylabel('Liquid Fraction'); ax.set_xlabel('Test Step'); ax.set_ylim(-0.05, 1.1)
    ax.set_title('PCM Liquid Fraction', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); _save(fig, f'prediction_{name}.png')

def plot_parity(res, name):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, (yt, yp, title, c) in zip(axes, [
        (res['y_true'], res['ens_std_pred'],   'Standard Ensemble', '#e07b39'),
        (res['y_true'], res['ens_ridge_pred'],  'Ridge Ensemble',    '#2ecc71'),
        (res['y_true'], res['ens_med_pred'],    'Median Ensemble',   '#9b59b6'),
    ]):
        n = min(len(yt), len(yp))
        ax.scatter(yt[:n], yp[:n], s=5, alpha=0.4, color=c, rasterized=True)
        lo, hi = min(yt[:n].min(), yp[:n].min()), max(yt[:n].max(), yp[:n].max())
        ax.plot([lo,hi],[lo,hi],'k--',lw=1.5,label='Ideal')
        m = metrics(yt[:n], yp[:n])
        r2_val = m.get('best_r2', m.get('r2', float('nan')))
        r2s = _safe_r2_fmt(r2_val)
        ax.set_xlabel('Actual T_battery (K)')
        ax.set_ylabel('Predicted T_battery (K)')
        ax.set_title(f'{title}\nR²={r2s}  MAE={m["mae"]:.4f} K', fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f'Parity — {name}', fontsize=13, fontweight='bold')
    plt.tight_layout(); _save(fig, f'parity_{name}.png')

def plot_uncertainty(res, name):
    fig, ax = plt.subplots(figsize=(13, 5))
    t   = np.arange(len(res['y_true']))
    ep  = res['ens_pred']
    stack = np.vstack([res['xgb_pred'], res['lstm_pred'], res['pima_pred']])
    band  = np.max(stack, axis=0) - np.min(stack, axis=0)
    ax.plot(t, res['y_true'],       'k-',  lw=1.8, label='Actual (clean)',         zorder=5)
    ax.plot(t, ep,                  color='#c0392b', lw=1.8, label='Best Ensemble')
    ax.plot(t, res['ens_std_pred'], color='#e67e22', lw=1.2, ls=':', label='Ens-Std')
    ax.plot(t, res['ens_ridge_pred'],color='#2ecc71',lw=1.2, ls=':', label='Ens-Ridge')
    ax.fill_between(t, ep-band, ep+band, color='#c0392b', alpha=0.12,
                    label='±Spread (model disagreement)')
    ax.set_title(f'{name} — Ensemble Uncertainty', fontweight='bold')
    ax.set_xlabel('Test Step'); ax.set_ylabel('T_battery (K)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); _save(fig, f'uncertainty_{name}.png')

def plot_ar_comparison_physics(pcm_name, scores):
    ars  = sorted(scores.keys())
    fig, axes = plt.subplots(1, 4, figsize=(17, 5))
    clrs = ['#4e8ac8', '#e07b39', '#5aaa68']
    for ax, (field, title) in zip(axes, [
        ('composite',    'Composite Score (↑)'),
        ('plateau_steps','Melting Plateau (steps)'),
        ('peak_rise_K',  'Peak Rise K (↓)'),
        ('dTdt_melting', 'Mean |dT/dt| in Melting (↓)'),
    ]):
        vals = [scores[ar].get(field, 0) for ar in ars]
        bars = ax.bar(ars, vals, color=clrs[:len(ars)], edgecolor='black', lw=0.8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.001,
                    f'{v:.4f}' if isinstance(v, float) else f'{v}',
                    ha='center', va='bottom', fontsize=9)
        ax.set_xlabel('Aspect Ratio'); ax.set_title(title, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
    plt.suptitle(f'Physics-Based AR Comparison — {pcm_name}', fontsize=13, fontweight='bold')
    plt.tight_layout(); _save(fig, f'ar_physics_comparison_{pcm_name}.png')

def plot_pcm_comparison_table(winners, best_pcm):
    rows = []
    for pcm_name, info in winners.items():
        res = info['res']
        # FIX: use true initial temperature, not test-set min
        T0 = res['T_initial']
        if 'T_battery_clean' in res['test_df'].columns:
            t_max = res['test_df']['T_battery_clean'].max()
        else:
            t_max = res['test_df']['T_battery'].max()
        peak_rise = max(0.0, t_max - T0)

        rows.append({
            'PCM':            pcm_name,
            'Best AR':        info['ar'],
            'Peak Rise (K)':  f"{peak_rise:.2f}",
            'Plateau Steps':  int(((res['test_df']['liquid_frac'] > 0.1) &
                                   (res['test_df']['liquid_frac'] < 0.9)).sum()),
            'Ens MAE (K)':    f"{res['ens_metrics']['mae']:.4f}",
            'Ens R²':         _safe_r2_fmt(res['ens_metrics'].get('best_r2',
                                            res['ens_metrics'].get('r2', float('nan')))),
            'Material Score': f"{info['material_score']:.3f}",
            'Total Score':    f"{0.60*info['material_score'] + 0.40*info['scores']['composite']:.3f}",
            'Status':         '★ BEST' if pcm_name == best_pcm else '',
        })
        
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(14, 2.5))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(10)
    tbl.scale(1.2, 1.8)
    for i in range(len(df.columns)):
        tbl[(0, i)].set_facecolor('#4472C4')
        tbl[(0, i)].set_text_props(weight='bold', color='white')
    plt.suptitle('Table — Best PCM Comparison (Physics & Battery-Activity Aware)',
                 fontsize=13, fontweight='bold', y=0.98)
    _save(fig, 'pcm_comparison_table.png')
    tprint("\n  TABLE: Best PCM Comparison")
    tprint(df.to_string(index=False))

def plot_re_air_visualization(fname='re_air_1000_visualization.png'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    params = [
        ('Nu_ext',          NU_EXT, 'Nusselt Number'),
        ('h_ext (W/m²K)',   H_EXT,  'Convection Coefficient'),
        ('R_ext (K/W)',     R_EXT,  'Thermal Resistance'),
    ]
    for ax, (label, val, title) in zip(axes, params):
        ax.bar([label], [val], color='steelblue', edgecolor='black', lw=1)
        ax.set_title(f'{title}\n{val:.4f}', fontweight='bold')
        ax.set_ylabel('Value'); ax.grid(axis='y', alpha=0.3)
    fig.text(0.5, 0.01,
             f'Re_air={RE_AIR:.0f}  |  Pr_air={PR_AIR:.3f}  |  D_cell={D_CELL*1000:.1f} mm  '
             f'|  L_cell={L_CELL*1000:.1f} mm  |  A_cell={A_CELL:.5f} m²',
             ha='center', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#f0f0f0', edgecolor='black'))
    plt.suptitle('External Convective Boundary Condition at Re = 1000',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    _save(fig, fname)

def plot_nca_runaway(nca_df, best_res, pcm_name, ar):
    fig, ax = plt.subplots(figsize=(13, 7))
    RUNAWAY_C = 60.0

    nca_runs = []
    for run in nca_df['run'].unique():
        run_df = nca_df[nca_df['run']==run].sort_values('TestTimes')
        t_raw  = run_df['TestTimes'].values
        T_raw  = run_df['SurfaceTempdegC'].values
        t_norm = (t_raw - t_raw[0]) / (t_raw[-1] - t_raw[0] + 1e-9)
        nca_runs.append((t_norm, T_raw))

    n        = min(len(r[0]) for r in nca_runs)
    t_common = np.linspace(0, 1, n)
    nca_interp = [np.interp(t_common, t_norm, T_raw) for t_norm, T_raw in nca_runs]
    nca_arr  = np.array(nca_interp)

    ax.fill_between(t_common, nca_arr.min(0), nca_arr.max(0),
                    alpha=0.2, color='gray', label='NCA bare-cell envelope')
    ax.plot(t_common, nca_arr.mean(0), 'k--', lw=2, label='NCA bare-cell mean')
    ax.axhline(RUNAWAY_C, color='red', ls='-', lw=2,
               label=f'Thermal runaway threshold ({RUNAWAY_C}°C)')

    t_model    = np.linspace(0, 1, len(best_res['ens_pred']))
    T_bat_pred = best_res['ens_pred'] - 273.15
    T_pcm_vals = (best_res['test_df']['T_pcm_clean'].values
                  [SEQ_STEPS : SEQ_STEPS + len(best_res['ens_pred'])] - 273.15) if 'T_pcm_clean' in best_res['test_df'].columns else (best_res['test_df']['T_pcm'].values[SEQ_STEPS : SEQ_STEPS + len(best_res['ens_pred'])] - 273.15)

    ax.plot(t_model, T_bat_pred, color='#c0392b', lw=2.5,
            label=f'{pcm_name} AR{ar} — Battery (ens.)')
    ax.plot(t_model, T_pcm_vals[:len(T_bat_pred)], color='#2980b9', lw=2, ls='-.',
            label=f'{pcm_name} AR{ar} — PCM surface')

    lf = best_res['test_df']['liquid_frac'].values[SEQ_STEPS : SEQ_STEPS + len(best_res['ens_pred'])]
    melt_idx = np.where((lf > 0.1) & (lf < 0.9))[0]
    if len(melt_idx) > 0:
        ax.axvspan(t_model[melt_idx[0]], t_model[melt_idx[-1]],
                   alpha=0.1, color='green', label='PCM melting zone')

    ax.set_xlabel('Normalized Discharge Time', fontweight='bold')
    ax.set_ylabel('Temperature (°C)', fontweight='bold')
    ax.set_title('PCM Thermal Protection vs Bare NCA Cell\n'
                 '(5C discharge, 25°C ambient, Re = 1000)', fontweight='bold')
    ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout(); _save(fig, 'nca_runaway_comparison.png')

def plot_table_model_comparison(results, fname='table13_model_comparison.png'):
    rows = []
    for key, r in results.items():
        for model_name, met in [
            ('TCN-GRU',   r['lstm_metrics']),
            ('XGBoost',   r['xgb_metrics']),
            ('PIMA',      r['pima_metrics']),
            ('Ens-Std',   r['ens_std_metrics']),
            ('Ens-Ridge', r['ens_ridge_metrics']),
            ('Ens-Med',   r['ens_med_metrics']),
            ('Best-Ens',  r['ens_metrics']),
        ]:
            r2_val = met.get('best_r2', met.get('r2', float('nan')))
            r2s = _safe_r2_fmt(r2_val)
            row = {
                'Case':    key,
                'Model':   model_name,
                'R²':      r2s,
                'RMSE (K)':f"{met['rmse']:.4f}",
                'MAE (K)': f"{met['mae']:.4f}",
                'MAPE (%)':f"{met['mape']:.4f}",
                'Skill':   f"{met.get('skill_vs_persist', float('nan')):.4f}",
            }
            if 'mae_melt' in met:
                row['MAE_melt (K)'] = f"{met['mae_melt']:.4f}"
            if 'skill_melt' in met:
                row['Skill_melt'] = f"{met['skill_melt']:.4f}"
            rows.append(row)
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(16, max(4, 0.28*len(df))))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9)
    tbl.scale(1.2, 1.5)
    for i in range(len(df.columns)):
        tbl[(0, i)].set_facecolor('#4472C4')
        tbl[(0, i)].set_text_props(weight='bold', color='white')
    plt.suptitle('Table 13 — Model Performance Comparison (v2.3 Clean-Target)',
                 fontsize=14, fontweight='bold', y=0.98)
    _save(fig, fname)
    tprint("\n  TABLE: Model Comparison")
    tprint(df.to_string(index=False))

def plot_table_pcm_properties(fname='table2_pcm_properties.png'):
    rows = []
    for pcm, p in PCM_PROPS.items():
        rows.append({
            'PCM':               pcm,
            'L_latent (J/kg)':   f"{p['L_latent']:.0f}",
            'Cp_liquid (J/kg·K)':f"{p['Cp_liquid']:.0f}",
            'Cp_solid (J/kg·K)': f"{p['Cp_solid']:.0f}",
            'k_liquid (W/m·K)':  f"{p['k_liquid']:.4f}",
            'ρ_liquid (kg/m³)':  f"{p['rho_liquid']:.0f}",
            'T_liquidus (K)':    f"{p['T_liquidus']:.2f}",
            'T_solidus (K)':     f"{p['T_solidus']:.2f}",
        })
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(10)
    tbl.scale(1.2, 1.8)
    for i in range(len(df.columns)):
        tbl[(0, i)].set_facecolor('#70AD47')
        tbl[(0, i)].set_text_props(weight='bold', color='white')
    plt.suptitle('Table 2 — Thermophysical Properties of PCMs',
                 fontsize=13, fontweight='bold', y=0.98)
    _save(fig, fname)
    tprint("\n  TABLE: PCM Properties")
    tprint(df.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 14. MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    import shutil
    # ── FULL CLEAN: wipe OUTPUT_DIR completely ─────────────────────────────
    if os.path.exists(OUTPUT_DIR):
        tprint(f"  [CLEAN] Wiping {OUTPUT_DIR} completely...")
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    # ───────────────────────────────────────────────────────────────────────

    tprint("="*70)
    tprint("  PCM Model 1 v2.3 — Kaggle-Fast | Clean-Target | Interrupt-Safe")
    tprint("  Single-dataset pre-train | SEQ=25 | Concise epoch logger")
    tprint(f"  XGBoost {xgb.__version__} | TF {tf.__version__} | Policy: {policy.name}")
    tprint("="*70)

    plot_table_pcm_properties()
    plot_re_air_visualization()

    global_scalers = fit_global_scalers(PCM_REGISTRY)
    pretrain_path = os.path.join(OUTPUT_DIR, 'tcn_gru_pretrain_ar04.keras')
    if not os.path.exists(pretrain_path):
        pretrain_model = pretrain_tcn_gru(PCM_REGISTRY, global_scalers)
        if pretrain_model is not None:
            del pretrain_model
            gc.collect()
            keras.backend.clear_session()
    else:
        tprint(f"  Pre-trained model found at {pretrain_path}; skipping pre-training.")

    per_ar: Dict[str, dict] = {}

    for pcm_name, ar_map in PCM_REGISTRY.items():
        for ar, path in ar_map.items():
            key = f"{pcm_name}_AR{ar}"
            checkpoint_path = os.path.join(OUTPUT_DIR, f'result_{key}.pkl')

            if os.path.exists(checkpoint_path):
                tprint(f"\n[RESUME] Loading completed result for {key}")
                try:
                    with open(checkpoint_path, 'rb') as f:
                        per_ar[key] = pickle.load(f)
                    res = per_ar[key]
                    xgb_ok = is_appropriate(res['xgb_metrics'])
                    lstm_ok = is_appropriate(res['lstm_metrics'])
                    pima_ok = is_appropriate(res['pima_metrics'])
                    ens_ok = is_appropriate(res['ens_metrics'])
                    if xgb_ok or lstm_ok or pima_ok or ens_ok:
                        plot_prediction_vs_actual(res, key)
                        plot_parity(res, key)
                        plot_uncertainty(res, key)
                        if lstm_ok and 'hist_df' in res and res['hist_df'] is not None and not res['hist_df'].empty:
                            plot_training_curves(res['hist_df'], key)
                    continue
                except Exception as load_err:
                    tprint(f"  [WARN] Failed to load checkpoint for {key}: {load_err}. Re-training.")

            if not os.path.exists(path):
                tprint(f"[SKIP] {path}"); continue

            try:
                res = run_ar(ar, path, pcm_name,
                             global_scalers=global_scalers,
                             pretrain_model_path=pretrain_path)
                per_ar[key] = res

                with open(checkpoint_path, 'wb') as f:
                    pickle.dump(res, f)

                xgb_ok = is_appropriate(res['xgb_metrics'])
                lstm_ok = is_appropriate(res['lstm_metrics'])
                pima_ok = is_appropriate(res['pima_metrics'])
                ens_ok = is_appropriate(res['ens_metrics'])

                if xgb_ok or lstm_ok or pima_ok or ens_ok:
                    plot_prediction_vs_actual(res, key)
                    plot_parity(res, key)
                    plot_uncertainty(res, key)
                    if lstm_ok and 'hist_df' in res and res['hist_df'] is not None and not res['hist_df'].empty:
                        plot_training_curves(res['hist_df'], key)

            except BaseException as e:
                tprint(f"\n[FATAL] {pcm_name} AR{ar}: {e}")
                import traceback; traceback.print_exc()
                if per_ar:
                    with open(os.path.join(OUTPUT_DIR, 'per_ar_partial.pkl'), 'wb') as f:
                        pickle.dump(per_ar, f)
                raise

    plot_table_model_comparison(per_ar)

    tprint("\n" + "="*70 + "\n  PHYSICS-BASED BEST AR SELECTION\n" + "="*70)
    for pcm_name in PCM_REGISTRY:
        best_ar, scores = select_best_ar_physics(per_ar, pcm_name)
        if best_ar:
            tprint(f"\n  [{pcm_name}] Best AR = {best_ar} (by cooling physics)")
            for ar, s in scores.items():
                tprint(f"    AR {ar}: composite={s['composite']:.3f}  "
                       f"plateau={s['plateau_steps']} steps  rise={s['peak_rise_K']:.2f} K  "
                       f"dT/dt={s['dTdt_melting']:.4f} K/s")
            plot_ar_comparison_physics(pcm_name, scores)

    tprint("\n" + "="*70 + "\n  BEST PCM OF THE THREE\n" + "="*70)
    best_pcm, winners = select_best_pcm_overall(per_ar)
    if best_pcm:
        plot_pcm_comparison_table(winners, best_pcm)
        best_ar  = winners[best_pcm]['ar']
        best_key = f"{best_pcm}_AR{best_ar}"
        best_res = per_ar[best_key]
        tprint(f"\n  ★ OVERALL WINNER: {best_pcm} AR {best_ar}")
        _peak_rise = max(0.0, best_res['test_df']['T_battery_clean'].max() - best_res['T_initial'])
        tprint(f"    Peak rise: {_peak_rise:.2f} K")
        tprint(f"    Ens MAE:   {best_res['ens_metrics']['mae']:.4f} K")
        ens_r2 = best_res['ens_metrics'].get('best_r2', best_res['ens_metrics']['r2'])
        tprint(f"    Ens R²:    {_safe_r2_fmt(ens_r2)}")

        tprint("\n" + "="*70 + "\n  NCA RUNAWAY IMPLEMENTATION\n" + "="*70)
        nca_df = load_nca()
        load_specs()
        if not nca_df.empty:
            plot_nca_runaway(nca_df, best_res, best_pcm, best_ar)
            safety = pcm_safety_check(nca_df)
            tprint("\n  PCM Safety Check:")
            for k, v in safety.items(): tprint(f"    {k}: {v}")
    else:
        tprint("  [WARN] Could not determine best PCM.")

    tprint("\n" + "="*70 + "\n  Model 1 v2.3 COMPLETE\n" + "="*70)
    tprint(f"  Outputs → {OUTPUT_DIR}")

  [CLEAN] Wiping /kaggle/working/output_v1 completely...
  PCM Model 1 v2.3 — Kaggle-Fast | Clean-Target | Interrupt-Safe
  Single-dataset pre-train | SEQ=25 | Concise epoch logger
  XGBoost 3.2.0 | TF 2.19.0 | Policy: float32
  Saved: /kaggle/working/output_v1/table2_pcm_properties.png

  TABLE: PCM Properties
          PCM L_latent (J/kg) Cp_liquid (J/kg·K) Cp_solid (J/kg·K) k_liquid (W/m·K) ρ_liquid (kg/m³) T_liquidus (K) T_solidus (K)
        RT-45          139700               2333              3028           0.2415              770         322.00        308.00
  Lauric_Acid          187210               2390              2180           0.1400              885         321.35        316.65
Palmitic_Acid          203400               2480              2200           0.2100              853         334.14        332.84
  Saved: /kaggle/working/output_v1/re_air_1000_visualization.png
  Fitting global scalers for TCN-GRU curriculum learning...
  Global scalers fitted on 169716 samples,

I0000 00:00:1781553888.482086      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781553888.488551      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  Pre-train model params: 79,610
  🚀 Training 80 epochs | monitor=val_T_battery_loss


I0000 00:00:1781553912.102315      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [░░░░░░░░░░░░░░░░░░░░] Epoch 001/80 | val_loss=0.1369 val_mae=0.5210 lr=2.00e-04 ✓
  [░░░░░░░░░░░░░░░░░░░░] Epoch 002/80 | val_loss=0.0359 val_mae=0.2639 lr=2.00e-04 ✓
  [░░░░░░░░░░░░░░░░░░░░] Epoch 003/80 | val_loss=0.0205 val_mae=0.1971 lr=2.00e-04 ✓
  [█░░░░░░░░░░░░░░░░░░░] Epoch 004/80 | val_loss=0.0302 val_mae=0.2414 lr=2.00e-04 -
  [█░░░░░░░░░░░░░░░░░░░] Epoch 005/80 | val_loss=0.0343 val_mae=0.2579 lr=2.00e-04 -
  [█░░░░░░░░░░░░░░░░░░░] Epoch 006/80 | val_loss=0.0330 val_mae=0.2530 lr=2.00e-04 -
  [█░░░░░░░░░░░░░░░░░░░] Epoch 007/80 | val_loss=0.0290 val_mae=0.2371 lr=2.00e-04 -
  [██░░░░░░░░░░░░░░░░░░] Epoch 008/80 | val_loss=0.0270 val_mae=0.2284 lr=2.00e-04 -
  [██░░░░░░░░░░░░░░░░░░] Epoch 009/80 | val_loss=0.0257 val_mae=0.2224 lr=1.00e-04 -
  ↻ LR dropped: 2.00e-04 → 1.00e-04
  [██░░░░░░░░░░░░░░░░░░] Epoch 010/80 | val_loss=0.0238 val_mae=0.2137 lr=1.00e-04 -
  [██░░░░░░░░░░░░░░░░░░] Epoch 011/80 | val_loss=0.0246 val_mae=0.2174 lr=1.00e-04 -
  [███░░░░░░░░░░░░░░░░░] Epoc